# 62 — Train + eval SID generator (W3)

Fine-tunes Qwen2.5-1.5B-Instruct + LoRA to emit 3 SID tokens for each W2 query.
Smoke mode: 200 steps, ~10 min. Full: 3 epochs over ~90K rows, ~3-4 hr on L4 / ~1-2 hr on Blackwell.

**Prereqs**: W1 + W2 artifacts on Drive at `/content/drive/MyDrive/recsys2026/sid/track_to_sid.parquet`
and `/content/drive/MyDrive/recsys2026/sid_training/{train,val}.parquet`. HF token in Colab Secrets as `HF_TOKEN`.


In [ ]:
# 1) GPU check.
!nvidia-smi | head -20

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

In [ ]:
# 3) HF auth — pull HF_TOKEN from Colab Secrets (same pattern as notebook 61).
# Setup: Colab → 🔑 Secrets pane → add `HF_TOKEN` with notebook access enabled.
import os
from google.colab import userdata
from huggingface_hub import login
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
print('HF auth OK')

In [ ]:
# 4) Mount Drive + symlink W1/W2 artifacts. Drive subdir naming matches notebook 61:
#   recsys2026_sid_cache              (W1 quantizer output)
#   recsys2026_sid_training_cache     (W2 generator training data)
#   recsys2026_sid_eval_cache         (W3 eval metrics — created here)
#   recsys2026_sid_generator_cache    (W3 LoRA + merged checkpoints)
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=False)

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
SYMLINKS = [
    ('sid',           f'{DRIVE_BASE}/recsys2026_sid_cache'),
    ('sid_training',  f'{DRIVE_BASE}/recsys2026_sid_training_cache'),
    ('sid_eval',      f'{DRIVE_BASE}/recsys2026_sid_eval_cache'),
    ('sid_generator', f'{DRIVE_BASE}/recsys2026_sid_generator_cache'),
]
for local_name, drive_path in SYMLINKS:
    dst = f'{LOCAL_BASE}/{local_name}'
    os.makedirs(drive_path, exist_ok=True)
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(drive_path, dst)
    print(f'symlink: {dst} -> {drive_path}')

# Verify W1 + W2 artifacts visible. If these fail, notebooks 60/61 didn't ship to Drive.
!ls -la experiments/cache/sid/track_to_sid.parquet
!ls -la experiments/cache/sid_training/train.parquet experiments/cache/sid_training/val.parquet

In [ ]:
# 5) Install/upgrade deps. Colab Pro base ships transformers + datasets + torch but
# their preinstalled torchao (0.10.0) is incompatible with recent peft (which requires
# torchao > 0.16). Upgrade torchao explicitly along with peft. Per project memory
# `feedback_colab_library_traps.md` trap #6.
!pip install -q -U \
    "peft>=0.10" \
    "transformers>=4.40" \
    "accelerate>=0.30" \
    "trl>=0.8" \
    "torchao>=0.17"
import transformers, peft, torch, torchao
print(f'transformers={transformers.__version__}, peft={peft.__version__}, '
      f'torch={torch.__version__}, torchao={torchao.__version__}')

In [ ]:
# 6) Pytest pre-flight on the SID modules. NO output truncation — if anything fails,
# we need to see the full traceback (--tb=long) and ALL collection errors.
!cd /content/recsys2026 && python -m pytest \
    tests/test_sid_vocab.py \
    tests/test_sid_training_format.py \
    tests/test_sid_inference.py \
    tests/test_sid_eval.py \
    -v --tb=long --no-header 2>&1

In [ ]:
# 7) TINY e2e training — 30 steps on 100 train rows, ~2-3 min on L4.
# Verifies vocab extension + LoRA wrap + grad flow + checkpoint save + Hub push end-to-end.
# Pushes BOTH the LoRA adapter AND the merged model (--merge) so cell 9 eval can pull it down.
#
# Logging: HF Trainer prints {loss, grad_norm, lr, epoch} every logging_steps=10 →
# you'll see ~3 metric lines for tiny + a tqdm progress bar updating in real time.
# Full stdout also tee'd to Drive at recsys2026_sid_generator_cache/tiny/training_log.txt
# so it survives Colab disconnects.
#
# Checkpointing: save_steps=30 (matches max_steps for tiny), so one checkpoint at end.
# Includes model + optimizer + scheduler + RNG state. Saved to the symlinked Drive dir.
# To resume after a crash: re-run this cell with `--resume` added to the args.
import os
os.makedirs('/content/drive/MyDrive/recsys2026_sid_generator_cache/tiny', exist_ok=True)
!cd /content/recsys2026 && python -u scripts/train_sid_generator.py \
    --tiny \
    --merge \
    --output-dir experiments/cache/sid_generator/tiny \
    --hub-repo OrRim123/recsys2026-sid-generator-qwen15b-tiny \
    2>&1 | tee /content/drive/MyDrive/recsys2026_sid_generator_cache/tiny/training_log.txt

In [ ]:
# 8) FULL training — SKIPPED in tiny e2e mode. Uncomment when ready for ~3-4 hr L4 run.
#
# Storage strategy: output_dir on EPHEMERAL Colab disk (NOT Drive symlinked).
# Trainer writes ~5 GB checkpoints there during training; HF Hub becomes the
# canonical store (~975 MB LoRA adapter + ~3 GB merged model). After the Hub
# push succeeds, --cleanup-after-push deletes the local copies, leaving Drive
# clean (only training_log.txt persists, ~5 MB).
#
# Drive footprint with this strategy: ~5 MB. HF Hub footprint: ~4 GB.
#
# Trade-off: Colab disconnect mid-training = checkpoints lost, no resume.
# Mitigation: --resume can pick up from a Hub-pushed checkpoint if you also pass
# --hub-strategy checkpoint (not enabled by default; adds ~5 GB upload per save).
# import os
# # Keep the training log on Drive (it's tiny + survives disconnects).
# os.makedirs('/content/drive/MyDrive/recsys2026_sid_generator_cache/full', exist_ok=True)
# !cd /content/recsys2026 && python -u scripts/train_sid_generator.py \
#     --output-dir /content/recsys2026_full_run \
#     --hub-repo OrRim123/recsys2026-sid-generator-qwen15b-v1 \
#     --merge \
#     --cleanup-after-push \
#     2>&1 | tee /content/drive/MyDrive/recsys2026_sid_generator_cache/full/training_log.txt
print('Cell 8 (full training) is skipped in tiny e2e mode. Uncomment to enable.')

In [ ]:
# 9) EVAL — constrained-beam decode over 30 val rows against the tiny model just pushed.
# Eval slice = 'all' (not 'raw') because tiny train_sample_n=100 may not have raw rows.
# nDCG will be ~0 here (tiny training can't actually learn SIDs in 30 steps); the goal is
# just to confirm the eval pipeline runs e2e without crashing.
!cd /content/recsys2026 && python scripts/eval_sid_generator.py \
    --model-id OrRim123/recsys2026-sid-generator-qwen15b-tiny-merged \
    --eval-slice all \
    --limit 30 \
    2>&1 | tail -30

In [ ]:
# 10) Read + display final gate metrics.
import json
m = json.load(open('experiments/cache/sid_eval/w3_eval_metrics.json'))
print(json.dumps(m, indent=2))
print()
print('=' * 60)
if m.get('gate_pass'):
    print(f"GATE PASS — mean nDCG@20={m['mean_ndcg_at_20']:.4f} "
          f"(threshold 0.12; delta vs Phase 0 = {m['delta_vs_phase0']:+.4f})")
else:
    print(f"GATE FAIL — mean nDCG@20={m['mean_ndcg_at_20']:.4f} (threshold 0.12)")
    if m.get('paired_bootstrap_ci'):
        ci = m['paired_bootstrap_ci']
        print(f"  paired-bootstrap CI = ({ci['lo']:.4f}, {ci['hi']:.4f})")
print('=' * 60)


## After the run

**Gate pass** (nDCG@20 ≥ 0.12 AND CI lower-bound > 0):
- Merged model is on Hub at `OrRim123/recsys2026-sid-generator-qwen15b-v1-merged`
- Proceed to W4: build `SID_GENERATOR` retrieval class + register `wrrf_bm25_dense_sid_v1`
- Update `MEMORY.md` with the W3 result file

**Gate fail**:
- If point-estimate is close (0.10-0.12) but CI includes 0: more training (5 epochs), check loss curve
- If point-estimate is low (<0.08): likely the W1 SID coarseness biting (3017 unique SIDs limits ceiling).
  Re-run W1 with smaller latent_dim (256→128) + larger codebook (256→512), then re-run W2 + W3.
- If loss diverged: drop LR to 1e-4, re-run.
